In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import time
import joblib
import pandas as pd

sys.path.append('../src')

## Генерация данных

In [3]:
data_mmlu = joblib.load('data/data_mmlu.pkl')
data_la = joblib.load('data/data_longanswer.pkl')
data_ner = joblib.load('data/data_ner.pkl')

In [4]:
data_mmlu.head()

,query,answer,true_answer,few_shot
0,\nКонтекст:\nЧто такое ипотека и ипотечный кре...,A) Жилое помещение,['D'],"({'role': 'user', 'content': 'Ты готов отвечат..."
1,\nКонтекст:\nКредитные карты.\n\nКакие действи...,D) 60 месяцев,['A'],"({'role': 'user', 'content': 'Ты готов отвечат..."
2,\nКонтекст:\nПриобретение готового жилья\nГото...,C) Отсутствие возможности оформления постоянно...,['C'],"({'role': 'user', 'content': 'Ты готов отвечат..."
3,\nКонтекст:\nПриобретение строящегося жилья.\n...,[C] 30%,['C'],"({'role': 'user', 'content': 'Ты готов отвечат..."
4,\nКонтекст:\nКоммерческая недвижимость\n\nУсло...,[A] Возраст заемщика от 21 до 75 лет.,['A'],"({'role': 'user', 'content': 'Ты готов отвечат..."


In [5]:
data_la.head()

,id_task,worker,answer,"Вы согласны с замечаниями и проблемами, описанными в CodeReview?",Все существенные проблемы перечислены в CodeReview?,"CodeReview не содержит токсичных комментариев (ненормативная лексика, оскорбительные высказывания или намеки)?",Ответ лаконичен и по существу?,"Ответ структурирован, все параметры раскрыты?",Как вы оцениваете качество CodeReview?,prompt,thema,few_shot
0,0-6d001ed68fbaeabb4040565953102b969d7cd91b.patch,Агулов Андрей Владимирович,CODE REVIEW\n1. Functionality: The code seems ...,3.0,3.0,3.0,3.0,3.0,4.0,You are the leader of the development team. Yo...,java,"({'role': 'user', 'content': 'Ты готов отвечат..."
1,0-6d001ed68fbaeabb4040565953102b969d7cd91b.patch,Королев Дмитрий Владиславович,CODE REVIEW\n1. Functionality: The code seems ...,2.0,3.0,3.0,3.0,3.0,4.0,You are the leader of the development team. Yo...,java,"({'role': 'user', 'content': 'Ты готов отвечат..."
2,0-6d001ed68fbaeabb4040565953102b969d7cd91b.patch,Цветков Даниил Святославович,CODE REVIEW\n1. Functionality: The code seems ...,2.0,3.0,3.0,2.0,3.0,4.0,You are the leader of the development team. Yo...,java,"({'role': 'user', 'content': 'Ты готов отвечат..."
3,0-6d001ed68fbaeabb4040565953102b969d7cd91b.patch,Магонов Никита Юрьевич,CODE REVIEW\n1. Functionality: The code seems ...,2.0,2.0,3.0,3.0,3.0,4.0,You are the leader of the development team. Yo...,java,"({'role': 'user', 'content': 'Ты готов отвечат..."
4,10-ea3bc88dc380cbcc3d756b89db306d53e26b2e5d.patch,Агулов Андрей Владимирович,CODE REVIEW\n1. Functionality: The added funct...,3.0,3.0,3.0,2.0,3.0,4.0,You are the leader of the development team. Yo...,java,"({'role': 'user', 'content': 'Ты готов отвечат..."


In [6]:
data_ner.head()

,id,extracted,target,text
0,8_1443820.tsv,"{'Drugname': [], 'Drugclass': [], 'Drugform': ...","{'Drugname': [], 'Drugclass': [], 'Drugform': ...",т. к. все таблетки разделил строго по времени ...
1,1_2555494.tsv,"{'Drugname': ['Римантадин'], 'Drugclass': [], ...","{'Drugname': ['Римантадин'], 'Drugclass': [], ...","Это старый-добрый Римантадин, только в сиропе.\n"
2,1_618967.tsv,"{'Drugname': [], 'Drugclass': [], 'Drugform': ...","{'Drugname': [], 'Drugclass': [], 'Drugform': ...",Но больше всего мне не понравился их размер.\n
3,5_873342.tsv,"{'Drugname': [], 'Drugclass': [], 'Drugform': ...","{'Drugname': [], 'Drugclass': [], 'Drugform': ...","У меня были ужасные боли, даже сидеть не могла.\n"
4,6_592814.tsv,"{'Drugname': [], 'Drugclass': [], 'Drugform': ...","{'Drugname': [], 'Drugclass': [], 'Drugform': ...",Пропив 5 упаковок это примерно 500-600 рублей ...


## Принципы работы с Scorer'ом

Главной задачей scorer'а является подсчет метрик качества.

Подсчет метрик реализован в методе `calc`, для подсчета необходим объект класса `Sampler`, и предполагается, что в нем есть DataFrame, необходимый для подсчета метрик, но реализация scorer'а "лежит на плечах" валидатора, поэтому логика в scorer'e может быть любой. 

Главное, чтобы метод `calc` возвращал словарь, где ключ - название метрики, значение - значение метрики.

Реализацию метрик необходимо выносить в `llm_val.utils.metrics`. В данном файле находится словарь, где ключ - название метрики, а значение - словарь с двумя обязательными ключами: `call`(`Callable`) и `is_singlecol`(`bool`):

```python
METRICS = {'single_mean': {'call': lambda x: np.mean(x.values), 'is_singlecol': True},
           'multicol_mean': {'call': lambda x: np.mean(x.values), 'is_singlecol': False}}
```

## Scorer

Все scorer'ы наследуются от абстрактного класса `Scorer`, у которого есть конструктор, в который передаются метрики, которые необходимо посчитать, и абстрактный метод `calc`, которые необходимо определить в своем scorer'е.

Метод `calc` должен(обязан) возвращать словарь, где ключ - название метрики, значение - значение метрики.

## UniversalScorer

`UniversalScorer` является примером реализации универсального scorer'а для датасетов MMLU и RuBQ. `UniversalScorer` достает из семплера DataFrame, необходимый для подсчета метрик, достает из него столбец `is_correct` и считает метрики. Если `metric_name is None` - считает все метрики, которые были переданы в конструктор. 

### Пример с MMLUSampler

In [7]:
from llm_val.sampler.MMLUSampler import MMLUSampler

sampler_mmlu = MMLUSampler(data_mmlu)

In [8]:
from llm_val.scorer.UniversalScorer import UniversalScorer
from llm_val.utils.metrics import METRICS

In [9]:
METRICS

{'single_mean': {'call': <function llm_val.utils.metrics.<lambda>(x)>,
  'is_singlecol': True},
 'multicol_mean': {'call': <function llm_val.utils.metrics.<lambda>(x)>,
  'is_singlecol': False}}

In [10]:
scorer = UniversalScorer(metrics=METRICS)

In [11]:
scorer.calc(sampler=sampler_mmlu, data_type='test', metric_name=None)

{'single_mean_metric_value': 0.6, 'multicol_mean': 0.6}

### Пример с LongAnswerSampler

In [12]:
from llm_val.sampler.LongAnswerSampler import LongAnswerSampler

sampler_la = LongAnswerSampler(data_la)

In [13]:
from llm_val.scorer.UniversalScorer import UniversalScorer
from llm_val.utils.metrics import METRICS

In [14]:
METRICS

{'single_mean': {'call': <function llm_val.utils.metrics.<lambda>(x)>,
  'is_singlecol': True},
 'multicol_mean': {'call': <function llm_val.utils.metrics.<lambda>(x)>,
  'is_singlecol': False}}

In [15]:
scorer = UniversalScorer(metrics=METRICS)

In [16]:
scorer.calc(sampler=sampler_la, data_type='test', metric_name=None)

{'single_mean_Вы согласны с замечаниями и проблемами, описанными в CodeReview?': 2.2,
 'single_mean_Все существенные проблемы перечислены в CodeReview?': 3.0,
 'single_mean_CodeReview не содержит токсичных комментариев (ненормативная лексика, оскорбительные высказывания или намеки)?': 3.0,
 'single_mean_Ответ лаконичен и по существу?': 3.0,
 'single_mean_Ответ структурирован, все параметры раскрыты?': 2.6,
 'single_mean_Как вы оцениваете качество CodeReview?': 3.4,
 'multicol_mean': 2.8666666666666667}

## NerScorer

`NerScorer` является специализированным scorer'ом для задач извлечения именованных сущностей (NER). 
`NerScorer` принимает на вход словарь с **ровно одной** метрикой, достает из семплера DataFrame с колонками `extracted` и `target`, содержащими извлеченные и целевые сущности соответственно.

### Пример с NerSampler

In [17]:
from llm_val.sampler.NerSampler import NerSampler

entity_types = ['Drugname', 'Drugclass', 'Drugform', 'DI', 'ADR', 'Finding']
sampler_ner = NerSampler(data_ner, entity_types)

In [18]:
from llm_val.scorer.NerScorer import NerScorer
from llm_val.utils.ner_metrics import NER_METRICS

ner_metric = {'ner_exact_match_micro_f1': NER_METRICS['ner_exact_match_micro_f1']}

scorer = NerScorer(metrics=ner_metric)
scorer.calc(sampler=sampler_ner, data_type='test', metric_name=None)

{'Drugname_f1': 0.9020270270270271,
 'Drugclass_f1': 0.9100529100529101,
 'Drugform_f1': 0.9362549800796812,
 'DI_f1': 0.6690909090909091,
 'ADR_f1': 0.5654205607476636,
 'Finding_f1': 0.2992125984251969,
 'micro_f1': 0.7540368006008262}